In [ ]:
!pip install requests python-dotenv tqdm langchain langchain-community langchain-core faiss-cpu sentence-transformers transformers torch huggingface-hub accelerate langchain-groq

In [ ]:
import os
from getpass import getpass

groq_key = getpass("Enter Groq API Key: ")
github_token = getpass("Enter GitHub Token: ")

os.environ["GROQ_API_KEY"] = groq_key
os.environ["GITHUB_TOKEN"] = github_token

print("✓ API keys configured (not stored in notebook)")

In [ ]:
import os
import requests
from typing import List, Dict, Tuple
from dataclasses import dataclass
import pickle

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_groq import ChatGroq

from sentence_transformers import CrossEncoder
from transformers import pipeline

from tqdm import tqdm

print("✓ All imports done")

## Configuration

In [ ]:
REPO_OWNER = "fastapi"
REPO_NAME = "fastapi"
REPO_FULL_NAME = f"{REPO_OWNER}/{REPO_NAME}"

MAX_COMMITS = 200
MAX_PULLS = 150
MAX_ISSUES = 150

FAISS_INDEX_PATH = "fastapi_vectorstore"

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
GROQ_MODEL = "llama-3.1-8b-instant"
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
NLI_MODEL = "facebook/bart-large-mnli"

BATCH_SIZE = 50
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150

GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print("✓ Constants set")

## Data Fetching

In [ ]:
class GitHubDataFetcher:
    def __init__(self, token, repo):
        self.headers = {"Authorization": f"token {token}"} if token else {}
        self.base_url = "https://api.github.com"
        self.repo = repo

    def get_commits(self, max_count):
        docs = []
        page = 1
        while len(docs) < max_count:
            url = f"{self.base_url}/repos/{self.repo}/commits"
            response = requests.get(url, headers=self.headers, params={"per_page": 100, "page": page})
            if response.status_code == 422 or response.status_code != 200:
                break
            data = response.json()
            if not data:
                break
            docs.extend(data)
            page += 1
        return docs[:max_count]

    def get_pull_requests(self, max_count, sort="created", direction="desc"):
        docs = []
        page = 1
        while len(docs) < max_count:
            url = f"{self.base_url}/repos/{self.repo}/pulls"
            params = {"state": "all", "per_page": 100, "page": page, "sort": sort, "direction": direction}
            response = requests.get(url, headers=self.headers, params=params)
            if response.status_code == 422 or response.status_code != 200:
                break
            data = response.json()
            if not data:
                break
            docs.extend(data)
            page += 1
        return docs[:max_count]

    def get_issues(self, max_count, sort="created", direction="desc"):
        docs = []
        page = 1
        while len(docs) < max_count:
            url = f"{self.base_url}/repos/{self.repo}/issues"
            params = {"state": "all", "per_page": 100, "page": page, "sort": sort, "direction": direction}
            response = requests.get(url, headers=self.headers, params=params)
            if response.status_code == 422 or response.status_code != 200:
                break
            data = response.json()
            if not data:
                break
            filtered = [i for i in data if "pull_request" not in i]
            docs.extend(filtered)
            page += 1
        return docs[:max_count]

print("✓ GitHubDataFetcher ready")

In [ ]:
class DocumentBuilder:
    def build_commit_document(self, item):
        sha = item["sha"]
        message = item["commit"]["message"]
        author = item["commit"]["author"]["name"]
        date = item["commit"]["author"]["date"]
        url = item["html_url"]

        content = f"Commit: {sha[:7]}\nAuthor: {author}\nDate: {date}\nMessage:\n{message}"

        return Document(
            page_content=content,
            metadata={"type": "commit", "sha": sha, "short_sha": sha[:7], "url": url}
        )

    def build_pull_request_document(self, item):
        pr_number = item["number"]
        title = item["title"]
        body = item["body"] or ""
        author = item["user"]["login"]
        created_at = item["created_at"]
        url = item["html_url"]

        content = f"Pull Request #{pr_number}: {title}\nAuthor: {author}\nCreated: {created_at}\nDescription:\n{body[:2000]}"

        return Document(
            page_content=content,
            metadata={"type": "pull_request", "pr": pr_number, "title": title, "url": url}
        )

    def build_issue_document(self, item):
        issue_number = item["number"]
        title = item["title"]
        body = item["body"] or ""
        author = item["user"]["login"]
        created_at = item["created_at"]
        url = item["html_url"]

        content = f"Issue #{issue_number}: {title}\nAuthor: {author}\nCreated: {created_at}\nDescription:\n{body[:2000]}"

        return Document(
            page_content=content,
            metadata={"type": "issue", "issue": issue_number, "title": title, "url": url}
        )

print("✓ DocumentBuilder ready")

In [ ]:
fetcher = GitHubDataFetcher(GITHUB_TOKEN, REPO_FULL_NAME)
builder = DocumentBuilder()

print("Fetching recent commits...")
recent_commits = fetcher.get_commits(150)
print(f"Got {len(recent_commits)} recent commits")

print("Fetching recent pull requests...")
recent_pulls = fetcher.get_pull_requests(100, sort="created", direction="desc")
print(f"Got {len(recent_pulls)} recent PRs")

print("Fetching oldest pull requests (founding decisions)...")
oldest_pulls = fetcher.get_pull_requests(100, sort="created", direction="asc")
print(f"Got {len(oldest_pulls)} oldest PRs")

print("Fetching recent issues...")
recent_issues = fetcher.get_issues(100, sort="created", direction="desc")
print(f"Got {len(recent_issues)} recent issues")

print("Fetching oldest issues (founding decisions)...")
oldest_issues = fetcher.get_issues(100, sort="created", direction="asc")
print(f"Got {len(oldest_issues)} oldest issues")

raw_commits = recent_commits

seen_pr_numbers = set()
deduped_pulls = []
for pr in recent_pulls + oldest_pulls:
    if pr["number"] not in seen_pr_numbers:
        deduped_pulls.append(pr)
        seen_pr_numbers.add(pr["number"])
raw_pulls = deduped_pulls

seen_issue_numbers = set()
deduped_issues = []
for issue in recent_issues + oldest_issues:
    if issue["number"] not in seen_issue_numbers:
        deduped_issues.append(issue)
        seen_issue_numbers.add(issue["number"])
raw_issues = deduped_issues

print(f"\nFinal counts — Commits: {len(raw_commits)}, PRs: {len(raw_pulls)}, Issues: {len(raw_issues)}")

In [ ]:
all_documents = []

print("Building commit documents...")
for commit in tqdm(raw_commits):
    all_documents.append(builder.build_commit_document(commit))

print("Building pull request documents...")
for pr in tqdm(raw_pulls):
    all_documents.append(builder.build_pull_request_document(pr))

print("Building issue documents...")
for issue in tqdm(raw_issues):
    all_documents.append(builder.build_issue_document(issue))

print(f"Total documents: {len(all_documents)}")

## Chunking & Embeddings

In [ ]:
class DocumentChunker:
    def __init__(self, chunk_size, overlap):
        self.chunk_size = chunk_size
        self.overlap = overlap

    def chunk_text(self, text):
        if len(text) <= self.chunk_size:
            return [text]
        chunks = []
        start = 0
        while start < len(text):
            end = start + self.chunk_size
            chunks.append(text[start:end])
            start += (self.chunk_size - self.overlap)
        return chunks

    def chunk_documents(self, documents):
        chunked = []
        for doc in documents:
            text_chunks = self.chunk_text(doc.page_content)
            for idx, chunk in enumerate(text_chunks):
                meta = doc.metadata.copy()
                meta["chunk_index"] = idx
                meta["total_chunks"] = len(text_chunks)
                chunked.append(Document(page_content=chunk, metadata=meta))
        return chunked

chunker = DocumentChunker(CHUNK_SIZE, CHUNK_OVERLAP)

print("Chunking documents...")
chunked_documents = chunker.chunk_documents(all_documents)
print(f"Total chunks: {len(chunked_documents)}")

In [ ]:
def create_embeddings_in_batches(documents, embedding_model, batch_size):
    noise_keywords = [
        "speciesist", "clarvia", "aeo badge", "autonomous agent integration",
        "sdk generation", "agent-readiness"
    ]

    filtered = []
    for doc in documents:
        content_lower = doc.page_content.lower()
        if not any(kw in content_lower for kw in noise_keywords):
            filtered.append(doc)

    print(f"Filtered {len(documents) - len(filtered)} noise documents")
    documents = filtered

    embeddings = HuggingFaceEmbeddings(
        model_name=embedding_model,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True}
    )

    total_batches = (len(documents) + batch_size - 1) // batch_size
    print(f"Processing {len(documents)} documents in {total_batches} batches...")

    vector_store = None

    for i in tqdm(range(0, len(documents), batch_size)):
        batch = documents[i:i + batch_size]
        if vector_store is None:
            vector_store = FAISS.from_documents(batch, embeddings)
        else:
            batch_store = FAISS.from_documents(batch, embeddings)
            vector_store.merge_from(batch_store)

    return vector_store

print("✓ Embedding function ready")

In [ ]:
if os.path.exists(FAISS_INDEX_PATH):
    print("Loading existing FAISS index...")
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True}
    )
    vector_store = FAISS.load_local(
        FAISS_INDEX_PATH,
        embeddings,
        allow_dangerous_deserialization=True
    )
    print("Index loaded successfully")
else:
    print("Creating new FAISS index...")
    vector_store = create_embeddings_in_batches(
        chunked_documents,
        EMBEDDING_MODEL,
        BATCH_SIZE
    )
    print("Saving FAISS index...")
    vector_store.save_local(FAISS_INDEX_PATH)
    print("Index saved successfully")

In [ ]:
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 15, "fetch_k": 50, "lambda_mult": 0.5}
)

print("✓ Retriever ready (k=15, fetch_k=50)")

## Reranking & Hallucination Guard

In [ ]:
class DocumentReranker:
    def __init__(self, model_name):
        self.model = CrossEncoder(model_name)

    def rerank(self, query, documents, top_k=5):
        if not documents:
            return []
        pairs = [[query, doc.page_content] for doc in documents]
        scores = self.model.predict(pairs)
        scored = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)
        return [doc for doc, score in scored[:top_k]]

reranker = DocumentReranker(RERANKER_MODEL)
print("✓ Reranker ready")

In [ ]:
class HallucinationGuard:
    def __init__(self, model_name):
        self.classifier = pipeline("zero-shot-classification", model=model_name)

    def verify_answer(self, answer, context, threshold=0.5):
        if not answer or not context:
            return False, 0.0
        result = self.classifier(
            context[:1000],
            candidate_labels=[answer[:200]],
            hypothesis_template="This text supports: {}"
        )
        score = result["scores"][0]
        return score >= threshold, score

hallucination_guard = HallucinationGuard(NLI_MODEL)
print("✓ Hallucination guard ready")

## RAG Chain & Response Formatting

In [ ]:
groq_llm = ChatGroq(
    model=GROQ_MODEL,
    groq_api_key=GROQ_API_KEY,
    temperature=0
)

print(f"✓ Groq LLM ready ({GROQ_MODEL})")

In [ ]:
def format_context(documents):
    parts = []
    for idx, doc in enumerate(documents, 1):
        meta = doc.metadata
        section = f"--- Source {idx} ---\nType: {meta.get('type', 'unknown')}\n"
        if meta.get("type") == "commit":
            section += f"Commit SHA: {meta.get('short_sha', 'N/A')}\nURL: {meta.get('url', 'N/A')}\n"
        elif meta.get("type") == "pull_request":
            section += f"PR Number: #{meta.get('pr', 'N/A')}\nURL: {meta.get('url', 'N/A')}\n"
        elif meta.get("type") == "issue":
            section += f"Issue Number: #{meta.get('issue', 'N/A')}\nURL: {meta.get('url', 'N/A')}\n"
        section += f"\nContent:\n{doc.page_content}\n"
        parts.append(section)
    return "\n".join(parts)


def extract_citations(documents):
    citations = {"commits": [], "pull_requests": [], "issues": []}
    seen_commits, seen_prs, seen_issues = set(), set(), set()

    for doc in documents:
        meta = doc.metadata
        if meta.get("type") == "commit":
            sha = meta.get("short_sha")
            if sha and sha not in seen_commits:
                citations["commits"].append({"sha": sha, "url": meta.get("url", "")})
                seen_commits.add(sha)
        elif meta.get("type") == "pull_request":
            pr_num = meta.get("pr")
            if pr_num and pr_num not in seen_prs:
                citations["pull_requests"].append({
                    "number": pr_num,
                    "title": meta.get("title", ""),
                    "url": meta.get("url", "")
                })
                seen_prs.add(pr_num)
        elif meta.get("type") == "issue":
            issue_num = meta.get("issue")
            if issue_num and issue_num not in seen_issues:
                citations["issues"].append({
                    "number": issue_num,
                    "title": meta.get("title", ""),
                    "url": meta.get("url", "")
                })
                seen_issues.add(issue_num)

    return citations


def clean_answer(text):
    text = text.replace("Pydantric", "Pydantic")
    text = text.replace("opeanapi", "OpenAPI")
    sentences = text.split('. ')
    last = sentences[-1].strip()
    if len(sentences) > 1 and (len(last) < 15 or last[-1] not in '.!?'):
        sentences = sentences[:-1]
    cleaned = '. '.join(sentences)
    if cleaned and cleaned[-1] not in '.!?':
        cleaned += '.'
    return cleaned


def compute_verification_score(answer, context):
    answer_words = set(answer.lower().split())
    context_words = set(context.lower().split())
    if not answer_words:
        return 0.0
    overlap = answer_words & context_words
    score = len(overlap) / len(answer_words)
    return round(min(score, 1.0), 2)

print("✓ Helper functions ready")

In [ ]:
@dataclass
class PatchContextResponse:
    answer: str
    citations: Dict
    is_verified: bool
    verification_score: float
    warning: str = ""

    def format_response(self):
        output = f"### Answer\n\n{self.answer}\n\n"

        if self.warning:
            output += f"⚠️ Warning: {self.warning}\n\n"

        output += "### Supporting Evidence\n\n"

        if self.citations["commits"]:
            output += "**Commits:**\n"
            for c in self.citations["commits"]:
                output += f"- [{c['sha']}]({c['url']})\n"
            output += "\n"

        if self.citations["pull_requests"]:
            output += "**Pull Requests:**\n"
            for pr in self.citations["pull_requests"]:
                output += f"- [PR #{pr['number']}: {pr['title']}]({pr['url']})\n"
            output += "\n"

        if self.citations["issues"]:
            output += "**Issues:**\n"
            for issue in self.citations["issues"]:
                output += f"- [Issue #{issue['number']}: {issue['title']}]({issue['url']})\n"
            output += "\n"

        output += f"*Verification Score: {self.verification_score:.2f}*"
        return output

print("✓ Response class ready")

In [ ]:
def answer_question(question):
    retrieved_docs = retriever.invoke(question)
    reranked_docs = reranker.rerank(question, retrieved_docs, top_k=5)
    context = format_context(reranked_docs)

    prompt = f"""You are a FastAPI repository historian. Answer using ONLY the context below.

Context:
{context[:2000]}

Question: {question}

Rules:
- Answer directly. Never say "Yes, the question is answerable" or similar meta-phrases.
- If the context lacks evidence, state exactly: "No direct evidence found in the retrieved commits/PRs/issues."
- Cite specific PR/Issue numbers when available.
- Keep the answer factual and concise.

Answer:"""

    raw_answer = groq_llm.invoke(prompt).content.strip()
    answer = clean_answer(raw_answer)

    is_verified, verification_score = hallucination_guard.verify_answer(answer, context)
    citations = extract_citations(reranked_docs)

    warning = ""
    if not is_verified:
        warning = "Low confidence answer. Verify against citations below."
    if "no direct evidence" in answer.lower():
        warning = "Limited evidence found in retrieved context."
    if not any(citations.values()):
        warning = "No specific citations found in retrieved documents."

    return PatchContextResponse(
        answer=answer,
        citations=citations,
        is_verified=is_verified,
        verification_score=verification_score,
        warning=warning
    )

print("✓ Answer function ready with Groq LLM")

## Try It Out

In [ ]:
import os
from langchain_groq import ChatGroq

groq_key = os.getenv("GROQ_API_KEY")

print(f"Key found: {groq_key[:10]}..." if groq_key else "❌ No key found")

try:
    test_llm = ChatGroq(model="llama-3.1-8b-instant", groq_api_key=groq_key, temperature=0)
    result = test_llm.invoke("Say hello in one word.")
    print(f"✓ Groq working: {result.content}")
except Exception as e:
    print(f"❌ Groq error: {str(e)}")

In [ ]:
questions = [
    "Why does FastAPI use Pydantic for validation?",
    "Why was dependency injection added to FastAPI?",
    "How does FastAPI achieve high performance?",
    "Why does FastAPI generate automatic API documentation?",
    "What is the purpose of the response_model parameter?"
]

for idx, question in enumerate(questions, 1):
    print(f"\n{'='*80}")
    print(f"Question {idx}/{len(questions)}: {question}")
    print(f"{'='*80}\n")
    try:
        response = answer_question(question)
        print(response.format_response())
    except Exception as e:
        print(f"Error: {str(e)}")
    print("\n" + "-"*80)

print("\n✅ All questions processed")

In [ ]:
import pickle

save_config = {
    "FAISS_INDEX_PATH": FAISS_INDEX_PATH,
    "EMBEDDING_MODEL": EMBEDDING_MODEL,
    "GROQ_MODEL": GROQ_MODEL,
    "RERANKER_MODEL": RERANKER_MODEL,
    "CHUNK_SIZE": CHUNK_SIZE,
    "CHUNK_OVERLAP": CHUNK_OVERLAP,
    "REPO_FULL_NAME": REPO_FULL_NAME
}

with open("config.pkl", "wb") as f:
    pickle.dump(save_config, f)

print("✓ config.pkl saved")
print("\nFiles ready for Streamlit deployment:")
print(f"  1. {FAISS_INDEX_PATH}/")
print(f"  2. config.pkl")
print(f"  3. app.py")
print(f"  4. utils.py")